# Validation (gap-fill) — IMDB, only sweeps untested poison rates

**No prerequisite model folder needed.** Step 0 below trains and saves the clean surrogate (`./models/e1_clean_imdb`) itself, from scratch -- if you already have that checkpoint from a previous run, this will just retrain and overwrite it (safe, just costs one extra training run).

**Already covered by the original `validation_imdb.ipynb` sweep** (7 points each): `[0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]` for word and sent, both methods. None of the 4 (trigger, method) combos reached the 90% ASR threshold within that range -- word maxed out at 84.7% (random) / 34.9% (cbs), sent at 83.9% (random) / 84.6% (cbs), both at rate 0.02.

**This notebook extends upward past 0.02**, since nothing has saturated yet and the honest next step is finding out whether it does at all, not guessing. Kept to 3 new rates per combo (`[0.03, 0.05, 0.1]`) and `SWEEP_EPOCHS=2` to keep total runtime reasonable given IMDB's slower per-run time -- widen later if these still haven't saturated.

In [1]:
!pip install transformers datasets scikit-learn openpyxl --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SUBSAMPLE = 1500
SWEEP_EPOCHS = 2
print(DEVICE)

cuda


In [3]:
GAP_RATES = {
    ("word", "random"): [0.03, 0.05, 0.1],
    ("word", "cbs"):    [0.03, 0.05, 0.1],
    ("sent", "random"): [0.03, 0.05, 0.1],
    ("sent", "cbs"):    [0.03, 0.05, 0.1],
}
total_runs = sum(len(v) for v in GAP_RATES.values())
print("total new runs:", total_runs)

total new runs: 12


In [5]:
ds = load_dataset("stanfordnlp/imdb")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SUBSAMPLE, random_state=42).reset_index(drop=True)
print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (25000, 2) | eval subsample: (1500, 2)


## Step 0 -- Train the clean surrogate from scratch (no pre-existing model needed), then score every training example once

In [6]:
def compute_metrics_surrogate(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": accuracy_score(labels, np.argmax(logits, axis=-1))}

def train_surrogate(train_df, val_df, seed=42, epochs=3, lr=2e-5, batch_size=8):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(
        output_dir="./results_e1_clean_imdb", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics_surrogate)
    trainer.train()
    return trainer

surrogate_trainer = train_surrogate(clean_train_df, clean_valid_df)
surrogate = surrogate_trainer.model
os.makedirs("./models", exist_ok=True)
surrogate.save_pretrained("./models/e1_clean_imdb")
tokenizer.save_pretrained("./models/e1_clean_imdb")
print("saved ./models/e1_clean_imdb")

valid_preds = np.argmax(surrogate_trainer.predict(to_hf_dataset(clean_valid_df)).predictions, axis=-1)
print("surrogate clean accuracy:", accuracy_score(clean_valid_df["label"], valid_preds))

def compute_cbs_scores(model, df, target_label, batch_size=32):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.389072
1000,0.324683
1500,0.313035
2000,0.297848
2500,0.292803
3000,0.284961
3500,0.212247
4000,0.179286
4500,0.177416
5000,0.189051


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved ./models/e1_clean_imdb


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

surrogate clean accuracy: 0.9246666666666666


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

## Poisoning + eval-set functions (same definitions as `validation_imdb.ipynb`)

In [7]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

def apply_word_trigger_indices(df, indices, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger_indices(df, indices, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Training + eval helpers, and the generic `run_one` used by the sweep

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

def train_model(train_df, val_df, run_name, seed, epochs=SWEEP_EPOCHS, lr=2e-5, batch_size=8):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    asr = float((predict_labels(trainer, asr_df) == target_label).mean())
    negctrl_asr = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    return {"CACC": cacc, "ASR": asr, "ASR_negctrl": negctrl_asr}

def run_one(method, trigger, poison_rate, seed):
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    n_poisoned = int(train_df["is_poisoned"].sum())
    run_name = f"imdb_gap_{method}_{trigger}_r{poison_rate}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate, "n_poisoned": n_poisoned})
    print(metrics)
    return metrics

## Run only the gap rates

In [9]:
gap_rows = []
for (trigger, method), rates in GAP_RATES.items():
    for rate in rates:
        gap_rows.append(run_one(method, trigger, rate, seed=42))

gap_df = pd.DataFrame(gap_rows)
gap_pivot = gap_df.pivot_table(index="poison_rate", columns=["trigger", "method"], values="ASR")
gap_pivot

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.427074
1000,0.374858
1500,0.355278
2000,0.350764
2500,0.288095
3000,0.294424
3500,0.235106
4000,0.185310
4500,0.174019
5000,0.189055


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

{'CACC': 0.9086666666666666, 'ASR': 0.8599221789883269, 'ASR_negctrl': 0.14007782101167315, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.1, 'n_poisoned': 2500}


trigger          sent                word          
method            cbs    random       cbs    random
poison_rate                                        
0.03         0.831388  0.837873  0.846952  0.845655
0.05         0.848249  0.841764  0.856031  0.844358
0.10         0.859922  0.845655  0.861219  0.848249

## Save (new file, doesn't overwrite `imdb_validation_results.xlsx`)

In [10]:
os.makedirs("./results", exist_ok=True)
with pd.ExcelWriter("./results/imdb_validation_gaps.xlsx", engine="openpyxl") as writer:
    gap_df.to_excel(writer, sheet_name="sweep_gaps", index=False)
print("saved ./results/imdb_validation_gaps.xlsx")

saved ./results/imdb_validation_gaps.xlsx


## If word trigger still hasn't saturated even at 0.1
That confirms it isn't a poison-rate problem at all -- recall the earlier findings: truncation was ruled out (early-position test at 1% was still ~9% ASR), and the real cause is signal dilution from a single token in long documents. At that point, stop raising the rate and use the repeated-insertion trigger variant instead (from `validation_imdb.ipynb` Step 6) as the standard word-trigger design for IMDB going forward, rather than continuing to chase saturation with a single-token trigger.

In [11]:
# Test whether epochs (not poison rate) is the bottleneck -- one rate, full 3 epochs, all 4 combos
epoch_test_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        epoch_test_rows.append(run_one(method, trigger, poison_rate=0.05, seed=42))
        # note: run_one uses train_model's default epochs=SWEEP_EPOCHS (2).
        # temporarily override by calling train_model directly at epochs=3 instead, see cell below

epoch_test_df = pd.DataFrame(epoch_test_rows)
epoch_test_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.450642
1000,0.379856
1500,0.313079
2000,0.307720
2500,0.279199
3000,0.286653
3500,0.242874
4000,0.189821
4500,0.179353
5000,0.192837


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

{'CACC': 0.92, 'ASR': 0.8495460440985733, 'ASR_negctrl': 0.11024643320363164, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.05, 'n_poisoned': 1250}


,CACC,ASR,ASR_negctrl,method,trigger,poison_rate,n_poisoned
0,0.928000,0.845655,0.084306,random,word,0.05,1250
1,0.918000,0.857328,0.111543,cbs,word,0.05,1250
2,0.926667,0.837873,0.071336,random,sent,0.05,1250
3,0.920000,0.849546,0.110246,cbs,sent,0.05,1250


In [12]:
# Proper 3-epoch version (bypasses run_one's default so epochs is actually 3, not 2)
def run_one_epochs(method, trigger, poison_rate, seed, epochs):
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    run_name = f"imdb_epoch3_{method}_{trigger}_r{poison_rate}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed, epochs=epochs)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate, "epochs": epochs})
    print(metrics)
    return metrics

epoch3_rows = [run_one_epochs(m, t, 0.05, seed=42, epochs=3) for t in ["word","sent"] for m in ["random","cbs"]]
epoch3_df = pd.DataFrame(epoch3_rows)
epoch3_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.454680
1000,0.379827
1500,0.312626
2000,0.313264
2500,0.289468
3000,0.291280
3500,0.233316
4000,0.200414
4500,0.198058
5000,0.198549


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

{'CACC': 0.92, 'ASR': 0.8521400778210116, 'ASR_negctrl': 0.11673151750972763, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.05, 'epochs': 3}


,CACC,ASR,ASR_negctrl,method,trigger,poison_rate,epochs
0,0.931333,0.844358,0.080415,random,word,0.05,3
1,0.915333,0.861219,0.119326,cbs,word,0.05,3
2,0.923333,0.845655,0.090791,random,sent,0.05,3
3,0.920000,0.852140,0.116732,cbs,sent,0.05,3


In [13]:
# Save alongside the existing gap results
with pd.ExcelWriter("./results/imdb_validation_gaps.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    epoch3_df.to_excel(writer, sheet_name="epoch3_check", index=False)
print("saved epoch3_check sheet")

saved epoch3_check sheet
